# MScFE 622 Stochastic Modeling - Group Work Project #2
## Step 1 - Data Preparation and Exploration

This notebook is the canonical technical analysis artifact. Step 1 constructs the maximum common daily sample for TLT, GLD, SPY, and VIX, reports the required ETF log returns and VIX first differences, and visualizes both series groups. Numerical results below are produced from the committed canonical Step 1 dataset; no values are estimated manually.

### Data source and financial interpretation

The project pipeline downloads daily Yahoo Finance **Adjusted Close** series for TLT, GLD, and SPY and the Yahoo Finance VIX series (`^VIX`), then retains dates on which all four prices are present. The VIX Index is designed as a forward-looking, approximately 30-day measure of expected S&P 500 volatility inferred from SPX option quotations (Cboe Global Markets). Whaley likewise describes the VIX as an option-market volatility benchmark and explains its financial interpretation (Whaley 98-105).

<!-- citekey: cboe2019vixfaq -->
<!-- citekey: whaley2009vix -->

**Source note.** Market observations: Yahoo Finance, downloaded by the project loader required by the assignment. VIX interpretation: Cboe Global Markets and Whaley (2009). Calculations and figures: project team.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import re

from IPython.display import Image, Markdown, display
import numpy as np
import pandas as pd

from vix_regime_allocation.plots import plot_etf_log_returns, plot_vix_change
from vix_regime_allocation.transform import OUTPUT_COLUMNS

repo_root = Path.cwd()
if not (repo_root / "data/processed/step1_data.csv").is_file():
    repo_root = repo_root.parent

data_path = repo_root / "data/processed/step1_data.csv"
references_path = repo_root / "reports/references.bib"
data = pd.read_csv(data_path, parse_dates=["Date"], index_col="Date")
data.index.name = "Date"

assert tuple(data.columns) == OUTPUT_COLUMNS
assert isinstance(data.index, pd.DatetimeIndex)
assert data.index.is_monotonic_increasing
assert not data.index.has_duplicates
assert not data.isna().any().any()
assert np.isfinite(data.to_numpy(dtype=float)).all()
assert (data[["TLT", "GLD", "SPY", "VIX"]] > 0.0).all().all()

data.head()

In [ ]:
sample_summary = pd.DataFrame(
    {
        "value": [
            data.index.min().date().isoformat(),
            data.index.max().date().isoformat(),
            len(data),
            int(data.isna().sum().sum()),
        ]
    },
    index=["start_date", "end_date", "observations", "missing_values"],
)
sample_summary

### ETF daily log returns

For ETF $i$ on trading row $t$, Step 1 uses the daily log return

$$
r_{i,t}=\ln\left(\frac{P_{i,t}}{P_{i,t-1}}\right),
$$

where $P_{i,t}$ is the adjusted close for ETF $i$ at row $t$. Missing common dates are removed **before** this lag is formed; there is no filling or interpolation. The first row is then removed because it has no preceding common-sample observation.

### Daily change in VIX

**Greek letter used below:** $\Delta$ - **delta**, pronounced *DEL-tuh*, meaning a finite change or difference.

The Step 1 VIX observation is the first difference

$$
\Delta VIX_t = VIX_t - VIX_{t-1}.
$$

This is a level change, not a percentage return. It measures how the option-implied volatility index changed between consecutive rows of the common sample.

In [ ]:
analysis_columns = [
    "TLT_log_return",
    "GLD_log_return",
    "SPY_log_return",
    "VIX_change",
]
descriptive_statistics = data[analysis_columns].describe().T
descriptive_statistics

In [ ]:
vix_extreme_date = data["VIX_change"].abs().idxmax()
vix_extreme_change = float(data.loc[vix_extreme_date, "VIX_change"])
etf_std = data[["TLT_log_return", "GLD_log_return", "SPY_log_return"]].std(ddof=1)
highest_std_series = str(etf_std.idxmax())
highest_std_value = float(etf_std.loc[highest_std_series])

display(
    Markdown(
        "**Computed Step 1 observations.** "
        f"The common sample contains **{len(data)}** rows from "
        f"**{data.index.min().date().isoformat()}** through "
        f"**{data.index.max().date().isoformat()}**. "
        f"The largest absolute daily VIX change occurs on "
        f"**{vix_extreme_date.date().isoformat()}** and equals "
        f"**{repr(vix_extreme_change)}** VIX points. "
        f"Among the three ETF log-return series, **{highest_std_series}** has the "
        f"largest sample daily standard deviation, **{repr(highest_std_value)}**, "
        "using `ddof=1`."
    )
)

### Required ETF-return figure

The next cell calls the shared project plotting function rather than recreating plotting logic inside the notebook.

**Source note.** Yahoo Finance adjusted-close observations; project-team log-return calculations and visualization.

In [ ]:
with TemporaryDirectory() as temporary_directory:
    etf_plot_path = Path(temporary_directory) / "step1_etf_log_returns.png"
    plot_etf_log_returns(data, etf_plot_path)
    display(Image(filename=str(etf_plot_path)))

### Required VIX-change figure

The plot shows the first difference of the VIX level. Positive observations indicate an increase in the index from the preceding common-sample row, while negative observations indicate a decrease. This sign interpretation follows directly from the Step 1 difference definition and is not a directional forecast for SPY.

**Source note.** Yahoo Finance VIX observations; project-team first-difference calculations and visualization. VIX financial interpretation follows Cboe Global Markets and Whaley (2009).

In [ ]:
with TemporaryDirectory() as temporary_directory:
    vix_plot_path = Path(temporary_directory) / "step1_vix_change.png"
    plot_vix_change(data, vix_plot_path)
    display(Image(filename=str(vix_plot_path)))

### Interpretation, assumptions, and limitations

Step 1 is deliberately descriptive. The ETF panels reveal when daily log-return fluctuations cluster or spike, while the VIX-change panel reveals when option-implied volatility moves sharply between common-sample rows. The computed statistics above quantify the sample span, the largest observed absolute VIX movement, and the relative daily dispersion of the three ETF return series without introducing a regime model.

Key assumptions and limitations are: (1) the analysis uses the Yahoo Finance fields returned by the specified loader; (2) only the maximum **common** dates are retained, so observations missing for any one series are removed before lags; (3) there is no interpolation or forward fill; (4) `VIX_change` is an arithmetic first difference in VIX points, not a VIX return; (5) Step 1 contains no causal, predictive, or out-of-sample claim; and (6) later Step 2 regime models must treat this cleaned dataset as their canonical input rather than constructing a different sample.

## Works Cited

The following MLA 9 entries are rendered programmatically from the cited keys in the canonical `reports/references.bib` registry.

In [ ]:
bibtex_text = references_path.read_text(encoding="utf-8")
entry_pattern = re.compile(r"@(\w+)\{([^,]+),(.*?)\n\}", re.DOTALL)
field_pattern = re.compile(r'^\s*(\w+)\s*=\s*"([^"]*)",?\s*$', re.MULTILINE)
references: dict[str, dict[str, str]] = {}
entry_types: dict[str, str] = {}
for entry_match in entry_pattern.finditer(bibtex_text):
    entry_type, key, body = entry_match.groups()
    if key in references:
        raise ValueError(f"Duplicate bibliography key: {key}")
    references[key] = dict(field_pattern.findall(body))
    entry_types[key] = entry_type

cited_keys = ["whaley2009vix", "cboe2019vixfaq"]
missing_keys = [key for key in cited_keys if key not in references]
assert not missing_keys, f"Unresolved citation keys: {missing_keys}"

whaley = references["whaley2009vix"]
cboe = references["cboe2019vixfaq"]
assert entry_types["whaley2009vix"] == "article"
assert entry_types["cboe2019vixfaq"] == "misc"

whaley_pages = whaley["pages"].replace("--", "-")
whaley_mla = (
    f'{whaley["author"]} "{whaley["title"]}." '
    f'*{whaley["journal"]}*, vol. {whaley["volume"]}, no. {whaley["number"]}, '
    f'{whaley["year"]}, pp. {whaley_pages}. doi:{whaley["doi"]}.'
)
cboe_mla = (
    f'{cboe["author"]}. "{cboe["title"]}." *{cboe["publisher"]}*, '
    f'{cboe["year"]}, {cboe["url"]}. Accessed 19 Aug. 2026.'
)
display(Markdown(f"- {whaley_mla}\n- {cboe_mla}"))